# 14장 · 분류 결과에 따라 다른 워크플로우 실행하기

영수증 위치와 인증 오류는 서로 다른 안내가 필요합니다. 먼저 분류 결과 한 단어를 얻고, 그 값을 받아 어느 작성 규칙을 적용할지는 파이썬 코드가 결정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 분류 결과 한 단어 읽기

재현을 위해 단어가 있는지 검사하는 작은 대역을 씁니다. 실제 분류 모델이 질문을 이해하는 과정을 흉내 내는 함수는 아닙니다.



**볼 곳**　‘문의 내용’ 안에서 ‘분류 결과’를 거쳐 어느 값이 오른쪽에 남는지 보세요.

**애니메이션의 의미**　billing이라는 결과가 다음 경로의 이름이 됩니다.


### 셀 01 · 영수증 질문 분류하기

키워드 영수증이 있고 오류는 없는 입력입니다.

In [1]:
def mock_classify(question):
    # 설명용 규칙입니다. 모델 판단을 재현하지 않습니다.
    billing = "영수증" in question
    technical = "오류" in question
    if billing == technical:
        return "clarify"
    return "billing" if billing else "technical"

print(mock_classify("영수증이 필요합니다"))

billing


billing이 나옵니다. 두 단어가 모두 있거나 모두 없으면 이 함수는 분류를 확정하지 않고 clarify를 반환합니다.

### 과제 02 · 복합 문의의 경로 예측하기

영수증과 오류가 모두 있는 한 문장을 만들어 분류값을 확인하세요.

In [2]:
# 복합 문의 하나를 mock_classify에 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(mock_classify(&quot;영수증 출력 중 오류가 났어요&quot;))</code></pre>
<p>실행 결과</p><pre><code>clarify</code></pre>
<p>두 조건이 모두 참이라 결과는 clarify입니다. 어떤 분야인지 더 물은 뒤에 작성할 수 있습니다.</p>
</details>

## 실제로 어느 단계가 호출됐나요?

허용하는 작성 규칙은 billing과 technical 둘입니다. 규칙을 찾을 수 있으면 한 번 쓰고, 그렇지 않으면 추가 설명을 요청합니다. 아래의 write는 문구 생성 과정을 대신하는 모의 함수입니다.



**볼 곳**　‘문의 분류’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘결제 처리’, 아래쪽은 ‘기술 처리’입니다.

**애니메이션의 의미**　들어온 문의에 맞는 한 경로만 실행합니다.


### 셀 03 · 경로 선택 함수 준비하기

제공 함수의 writes는 최종 안내 작성 함수를 실제로 몇 번 불렀는지 표시합니다.

분류기가 빈 문자열을 그대로 반환하면 작성 규칙을 찾지 못해 needs_clarification으로 끝납니다. SDK 연결부가 먼저 빈 응답을 예외로 처리하면 route_request는 classification_error를 기록합니다. 어느 쪽도 작성 함수를 부르지 않습니다.

In [3]:
WRITING_RULES = {"billing": "결제 문의의 접수 안내를 작성하세요.", "technical": "기술 문의의 확인 항목을 작성하세요."}

class ClassificationRefusalError(ValueError):
    """명시적 거절을 정상적인 추가 질문과 구분하는 예외입니다."""

def route_request(classify, write, question):
    if not question.strip():
        raise ValueError("빈 질문")
    try:
        route = classify(question)
    except ClassificationRefusalError as exc:
        return {"route": "clarify", "reason": "classification_refusal",
                "error_type": type(exc).__name__, "answer": "분류 요청이 거절되었습니다. 요청 내용을 확인하세요.", "writes": 0}
    except Exception as exc:
        return {"route": "clarify", "reason": "classification_error",
                "error_type": type(exc).__name__, "answer": "분류에 실패했습니다. 잠시 뒤 다시 확인하세요.", "writes": 0}
    if route not in WRITING_RULES:
        return {"route": "clarify", "reason": "needs_clarification",
                "answer": "결제 문의인지 기술 문제인지 구체적으로 알려 주세요.", "writes": 0}
    answer = write(WRITING_RULES[route], question)
    return {"route": route, "reason": "classified", "answer": answer, "writes": 1}

def mock_write(instruction, question):
    return "[모의 안내] " + instruction

result = route_request(mock_classify, mock_write, "인증 오류입니다")
print(result["route"], result["writes"])

technical 1


technical 1은 기술 문의 규칙으로 작성 함수를 한 번 실행했다는 뜻입니다. 분류 호출은 writes에 들어가지 않습니다.

### 셀 04 · 분류 요청 자체가 실패한 경우

분류기가 답을 돌려주기 전에 시간 초과가 났다고 가정합니다.

In [4]:
def timed_out(question):
    raise TimeoutError("분류 대기 시간 초과")

failed = route_request(timed_out, mock_write, "영수증이 필요합니다")
print(failed["reason"], failed["writes"])

classification_error 0


classification_error 0입니다. 문의가 모호했기 때문이 아니라 분류 과정에서 실패해 작성까지 가지 못했습니다.

## 사용자에게 무엇을 다시 물을까요?

복합 문의와 시스템 오류의 화면 문구가 같으면 사용자가 올바른 다음 행동을 고르기 어렵습니다. 한 건의 분류 기록에서 이유와 안내를 함께 읽습니다.



**볼 곳**　‘모호한 문의’가 가운데 ‘분류 보류’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　분류가 모호하면 사용자에게 문의 종류를 다시 묻습니다.


### 과제 05 · 모호한 요청의 다음 안내

‘도와주세요’를 route_request에 넣고 기록에서 사용자에게 보여 줄 답변을 꺼내세요.

In [5]:
# 요청을 처리한 뒤 answer를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>ambiguous = route_request(mock_classify, mock_write, &quot;도와주세요&quot;)
print(ambiguous[&quot;answer&quot;])</code></pre>
<p>실행 결과</p><pre><code>결제 문의인지 기술 문제인지 구체적으로 알려 주세요.</code></pre>
<p>결제 문의인지 기술 문제인지 구체적으로 알려 달라는 안내입니다. 분류 실패의 ‘잠시 뒤 다시 확인’과는 다음 행동이 다릅니다.</p>
</details>

## 돌아보기

정상 분류, 복합 문의, 분류 시간 초과 중 한 건을 골라 어디서 분기하는지 설명해 보세요. reason으로 분기 이유를, writes로 작성 함수를 몇 번 불렀는지 확인할 수 있습니다.

### 분류 뒤의 실행 결정



**볼 곳**　분류값이 정해진 뒤 어느 실행 기록에만 호출이 추가되는지 보세요.

**애니메이션의 의미**　분류 결과가 다음 경로를 결정합니다. 한 문의에서 선택한 처리 단계만 실행됩니다.
